In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn import tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

%matplotlib inline

In [ ]:
data = pd.read_csv("water_potability.csv")

In [ ]:
print("Dataset loaded successfully!")
print("Shape before cleaning:", data.shape)
print("\nMissing values before cleaning:\n", data.isnull().sum())

In [ ]:
imputer = KNNImputer(n_neighbors=5)
data_imputed = pd.DataFrame(imputer.fit_transform(data), columns=data.columns)
print("\nMissing values handled using KNN Imputer.")
print("Missing values after imputation:\n", data_imputed.isnull().sum())

In [ ]:
for col in data_imputed.columns[:-1]:  # Skip 'Potability' column
    Q1 = data_imputed[col].quantile(0.25)
    Q3 = data_imputed[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR
    data_imputed[col] = np.clip(data_imputed[col], lower_limit, upper_limit)

print("\nOutliers capped using IQR method.")

In [ ]:
data_imputed['ph'] = data_imputed['ph'].clip(lower=0, upper=14)

In [ ]:
scaler = StandardScaler()
scaled_features = scaler.fit_transform(data_imputed.drop('Potability', axis=1))

data_scaled = pd.DataFrame(scaled_features, columns=data_imputed.columns[:-1])
data_scaled['Potability'] = data_imputed['Potability']

print("\nFeatures scaled using StandardScaler.")

In [ ]:
before = data_scaled.shape[0]
data_scaled = data_scaled.drop_duplicates()
after = data_scaled.shape[0]
print(f"\nRemoved {before - after} duplicate rows (if any).")

In [ ]:
data_scaled.to_csv("cleaned_water_potability.csv", index=False)
print("\nCleaned dataset saved as 'cleaned_water_potability.csv'")

In [ ]:
print("\nFinal Dataset Summary:")
print(data_scaled.describe())
print("\nFinal shape:", data_scaled.shape)

In [ ]:
data = pd.read_csv("cleaned_water_potability.csv")

In [ ]:
X = data.drop('Potability', axis=1)
y = data['Potability']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 20% for testing
    random_state=42,     # Reproducible split
    stratify=y           # Maintain class balance
)

In [ ]:
print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)
print("Training labels:", y_train.shape)
print("Testing labels:", y_test.shape)

In [ ]:
def evaluate_model(y_true, y_pred, y_proba=None, model_name="Model"):
    print(f"\n Results for {model_name}")
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    if y_proba is not None:
        auc = roc_auc_score(y_true, y_proba)
        print(f"ROC-AUC:   {auc:.4f}")

    print("\nClassification Report:\n", classification_report(y_true, y_pred, zero_division=0))
    print()
    # Confusion Matrix
    plt.figure(figsize=(5,4))
    sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt="d", cmap="Blues",
                xticklabels=["Not Potable","Potable"], yticklabels=["Not Potable","Potable"])
    plt.title(f"{model_name} - Confusion Matrix")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.show()
    print()
    # ROC Curve
    if y_proba is not None:
        fpr, tpr, _ = roc_curve(y_true, y_proba)
        plt.figure(figsize=(6,4))
        plt.plot(fpr, tpr, label=f"{model_name} (AUC={roc_auc_score(y_true, y_proba):.3f})", color="blue")
        plt.plot([0,1],[0,1],'--',color='gray')
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title(f"{model_name} - ROC Curve")
        plt.legend()
        plt.show()

Logistic Regression

In [ ]:
# Initialize model
lr = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)

# Train the model
lr.fit(X_train, y_train)

# Predictions
y_pred_lr = lr.predict(X_test)
y_proba_lr = lr.predict_proba(X_test)[:,1]

# Evaluate
evaluate_model(y_test, y_pred_lr, y_proba_lr, "Logistic Regression")

Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight='balanced'
)

# Train the model
rf.fit(X_train, y_train)

# Predictions
y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:,1]

# Evaluate
evaluate_model(y_test, y_pred_rf, y_proba_rf, "Random Forest")
print()

# Feature Importance
importances = rf.feature_importances_
feat_imp = pd.Series(importances, index=X_train.columns).sort_values(ascending=True)
plt.figure(figsize=(7,5))
feat_imp.plot(kind='barh', color='forestgreen')
plt.title("Random Forest - Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()


In [ ]:
xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)

# Train the model
xgb.fit(X_train, y_train)

# Predictions
y_pred_xgb = xgb.predict(X_test)
y_proba_xgb = xgb.predict_proba(X_test)[:,1]

# Evaluate
evaluate_model(y_test, y_pred_xgb, y_proba_xgb, "XGBoost")
print()

# Feature Importance
importances = xgb.feature_importances_
feat_imp = pd.Series(importances, index=X_train.columns).sort_values(ascending=True)
plt.figure(figsize=(7,5))
feat_imp.plot(kind='barh', color='darkorange')
plt.title("XGBoost - Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()


Decision Tree

In [ ]:
# Initialize model
dt = DecisionTreeClassifier(
    criterion='gini',      # or 'entropy'
    max_depth=None,        # let it grow fully (you can limit depth to prevent overfitting)
    random_state=42,
    class_weight='balanced'
)

# Train the model
dt.fit(X_train, y_train)

# Predictions
y_pred_dt = dt.predict(X_test)
y_proba_dt = dt.predict_proba(X_test)[:,1]

# Evaluate
evaluate_model(y_test, y_pred_dt, y_proba_dt, "Decision Tree")
print()
# Feature Importance
importances = dt.feature_importances_
feat_imp = pd.Series(importances, index=X_train.columns).sort_values(ascending=True)
plt.figure(figsize=(7,5))
feat_imp.plot(kind='barh', color='olive')
plt.title("Decision Tree - Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

KNN Algorithm

In [ ]:
# Initialize KNN model
knn = KNeighborsClassifier(
    n_neighbors=5,   # you can tune this value (3, 5, 7, etc.)
    metric='minkowski',
    p=2              # p=2 means Euclidean distance
)

# Train the model
knn.fit(X_train, y_train)

# Predictions
y_pred_knn = knn.predict(X_test)
y_proba_knn = knn.predict_proba(X_test)[:,1]

# Evaluate
evaluate_model(y_test, y_pred_knn, y_proba_knn, "K-Nearest Neighbors")

In [ ]:


# Features in correct order
features = [
    "ph", "Hardness", "Solids", "Chloramines", "Sulfate",
    "Conductivity", "Organic_carbon", "Trihalomethanes", "Turbidity"
]

print("Enter 9 water quality values (space separated) in this order:")
print("ph Hardness Solids Chloramines Sulfate Conductivity Organic_carbon Trihalomethanes Turbidity")
print("Example: 7.0 200 21000 7.5 330 420 14 66 3.9")

# Take user input
user_input = list(map(float, input("\nYour values: ").split()))

if len(user_input) != 9:
    print("\n Please enter exactly 9 values.")
else:
    # Convert input to DataFrame with feature names
    sample = pd.DataFrame([user_input], columns=features)

    print("\n Model Predictions:")
    print("-" * 45)

    # Logistic Regression
    pred_lr = lr.predict(sample)[0]
    print(f" Logistic Regression: {' Potable' if pred_lr==1 else ' Not Potable'}")

    # Random Forest
    pred_rf = rf.predict(sample)[0]
    print(f" Random Forest: {' Potable' if pred_rf==1 else ' Not Potable'} (Final Model )")

    # XGBoost
    pred_xgb = xgb.predict(sample)[0]
    print(f" XGBoost: {'Potable' if pred_xgb==1 else ' Not Potable'}")

    # KNN
    pred_knn = knn.predict(sample)[0]
    print(f" KNN: {' Potable' if pred_knn==1 else ' Not Potable'}")

    # Decision Tree
    pred_dt = dt.predict(sample)[0]
    print(f" Decision Tree: {' Potable' if pred_dt==1 else ' Not Potable'}")

    print("-" * 45)

    # Final Decision based on Random Forest (best model)
    print("\n FINAL DECISION (Based on Best Model - Random Forest):")
    if pred_rf == 1:
        print("Water is POTABLE (Safe for drinking)")
    else:
        print(" Water is NOT POTABLE (Unsafe for drinking)")


Enter 9 water quality values (space separated) in this order:
ph Hardness Solids Chloramines Sulfate Conductivity Organic_carbon Trihalomethanes Turbidity
Example: 7.0 200 21000 7.5 330 420 14 66 3.9
